In [ ]:
import serial
import time
import math
import numpy as np

# ==========================================
# CONFIGURATION
# ==========================================
PORT = "COM17"        # Change to your port
BAUD = 115200

# Robot Geometry (mm)
L1 = 26.0    # Coxa
L2 = 57.0    # Femur
L3 = 122.0   # Tibia
HIP_RADIUS = 137.5

# Servo Config
SERVO_MID_US = 1650

# CLAMP LIMITS (not range, hard stops)
HIP_MIN_US = 1000
HIP_MAX_US = 2300

KNEE_MIN_US = 1200
KNEE_MAX_US = 2300

FOOT_MIN_US = 1000
FOOT_MAX_US = 2050

# Leg Mounting Angles (Radians)
# Standard 6-leg radial symmetry: 0, 60, 120, 180, 240, 300
LEG_MOUNT_ANGLES = np.deg2rad([0, 60, 120, 180, 240, 300])

# ==========================================
# HELPER FUNCTIONS
# ==========================================

def angle_to_us(angle_deg, clamp_min, clamp_max):
    """
    Convert angle (degrees) to microseconds using fixed rate:
    0 deg = 1650 us
    +/- 90 deg = +/- 1000 us (11.11 us/deg)
    Then CLAMPS to [clamp_min, clamp_max]
    """
    rate = 1000.0 / 90.0 
    us = SERVO_MID_US + (angle_deg * rate)
    us = max(clamp_min, min(clamp_max, us))
    return int(us)

def solve_fk_3d(base_yaw, angle1, angle2, angle3, l1, l2, l3):
    """
    Forward Kinematics: Calculates tip coordinate for a 3DOF leg.
    Takes SERVO angles in RADIANS (0 = 1650us).
    Returns (x, y, z) in LEG FRAME.
    """
    # Adjustment for Tibia: Servo 0 (1650) = 90 degree bend (Geometric -90/PI2)
    # So Geometric Angle = Servo Angle - PI/2
    angle3_geom = angle3 - (math.pi / 2.0)
    
    # Distance from hip pivot (origin) projected on XY plane
    # r = L1 + L2*cos(a2) + L3*cos(a2+a3)
    r = l1 + l2 * math.cos(angle2) + l3 * math.cos(angle2 + angle3_geom)
    
    # Z height
    # z = L2*sin(a2) + L3*sin(a2+a3)
    z = l2 * math.sin(angle2) + l3 * math.sin(angle2 + angle3_geom)
    
    # X, Y coordinates
    x = r * math.cos(base_yaw)
    y = r * math.sin(base_yaw)
    
    return x, y, z

def solve_ik_3d(x, y, z, l1, l2, l3):
    """
    Calculates SERVO angles (radians) for a 3DOF leg.
    x, y, z: Target coordinates in LEG FRAME 
    Returns: (base_yaw, angle1, angle2, angle3) or None
    """
    base_yaw = math.atan2(y, x)
    r_global = math.hypot(x, y)
    r_target = r_global - l1
    d_sq = r_target**2 + z**2
    d = math.sqrt(d_sq)
    
    if d > l2 + l3 or d < abs(l2 - l3):
        return None
    
    cos_knee = (l2**2 + l3**2 - d_sq) / (2 * l2 * l3)
    cos_knee = max(-1.0, min(1.0, cos_knee))
    knee_interior = math.acos(cos_knee)
    
    # Tibia Angle
    # Geometric: 0 = straight, -90 = down.
    # Servo: 1650(0) = down (90 deg bend).
    # Geom Angle = -(PI - Interior)
    # Servo Angle = Geom + PI/2; 
    angle3_geom = -(math.pi - knee_interior)
    angle3 = angle3_geom + (math.pi / 2.0)
    
    az = math.atan2(z, r_target)
    cos_offset = (l2**2 + d_sq - l3**2) / (2 * l2 * d)
    cos_offset = max(-1.0, min(1.0, cos_offset))
    femur_offset = math.acos(cos_offset)
    angle2 = az + femur_offset
    
    angle1 = 0.0
    
    return base_yaw, angle1, angle2, angle3

def get_servo_us_for_leg(leg_index, target_x_global, target_y_global, target_z_global):
    mount_angle = LEG_MOUNT_ANGLES[leg_index]
    hip_x = HIP_RADIUS * math.cos(mount_angle)
    hip_y = HIP_RADIUS * math.sin(mount_angle)
    dx = target_x_global - hip_x
    dy = target_y_global - hip_y
    dz = target_z_global
    
    local_x = dx * math.cos(-mount_angle) - dy * math.sin(-mount_angle)
    local_y = dx * math.sin(-mount_angle) + dy * math.cos(-mount_angle)
    local_z = dz
    
    result = solve_ik_3d(local_x, local_y, local_z, L1, L2, L3)
    if result is None:
        print(f"Leg {leg_index} Unreachable")
        return 1650, 1650, 1650
        
    base_yaw, _, angle2, angle3 = result
    
    deg_hip   = math.degrees(base_yaw)
    deg_femur = math.degrees(angle2)
    deg_tibia = math.degrees(angle3)

    us_hip = angle_to_us(deg_hip, HIP_MIN_US, HIP_MAX_US)
    us_knee = angle_to_us(deg_femur, KNEE_MIN_US, KNEE_MAX_US)
    us_tibia = angle_to_us(deg_tibia, FOOT_MIN_US, FOOT_MAX_US)
    
    return us_hip, us_knee, us_tibia


# ==========================================
# VERIFICATION: LOOPBACK TEST
# ==========================================
print("=== Starting IK/FK Verification ===")
print("1. Calculating Tip Coordinates for Nominal Angles (0 deg, 1650 us)")
print("   (Expect Tibia Down / 90 deg bend)")

# Nominal SERVO Angles in Radians (0 corresponds to 1650us)
nom_yaw = 0.0
nom_femur = 0.0
nom_tibia = 0.0

# Calculate FK in Local Leg Frame
loc_x, loc_y, loc_z = solve_fk_3d(nom_yaw, nom_femur, nom_femur, nom_tibia, L1, L2, L3)
print(f"   Nominal Local Tip: ({loc_x:.2f}, {loc_y:.2f}, {loc_z:.2f})")

print("\n2. Feeding Coordinates Back into IK")
# Global Targets for each leg using these local coords
all_servo_us = []

for i in range(6):
    mount = LEG_MOUNT_ANGLES[i]
    
    # Convert Local -> Global for the test input
    hip_x = HIP_RADIUS * math.cos(mount)
    hip_y = HIP_RADIUS * math.sin(mount)
    
    gx = hip_x + loc_x * math.cos(mount) - loc_y * math.sin(mount)
    gy = hip_y + loc_x * math.sin(mount) + loc_y * math.cos(mount)
    gz = loc_z 
    
    print(f"   Leg {i} Global Target: ({gx:.2f}, {gy:.2f}, {gz:.2f})")
    
    # Run IK
    us1, us2, us3 = get_servo_us_for_leg(i, gx, gy, gz)
    all_servo_us.extend([us1, us2, us3])
    
    print(f"   Leg {i} Result US: Hip={us1}, Knee={us2}, Foot={us3}")
    
    # Check Method
    if us1 == 1650 and us2 == 1650 and us3 == 1650:
        print(f"   -> MATCH (1650)")
    else:
        print(f"   -> MISMATCH! Expected 1650")
